# Visualize the OC + HTN gene network

**Interactive HTML views** (vis-network, no extra install needed) to explore in the browser: zoom, drag, hover a gene for details.

Colours: **aqua** = ovarian cancer, **red** = hypertension. A gene in both diseases is drawn as a half/half **pie** of the two colours.

In [27]:
import json
import math
from pathlib import Path
from urllib.parse import quote

import networkx as nx
import numpy as np
import pandas as pd

CUTOFF = 0.1
DISEASE_A, DISEASE_B = "OC", "HTN"

INTERACTOME_FILE = "../hypertention - breast cancer/outputs/interactome/interactome_both_symbols.csv"
OUT_DIR = Path("outputs") / f"cutoff_{CUTOFF}" / f"network_{DISEASE_A}_{DISEASE_B}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DISEASE_FILES = {
    "OC": "inputs/diseases/ovarian cancer-MONDO_0008170.tsv",
    "HTN": "inputs/diseases/hypertensive disorder-MONDO_0005044.tsv",
    "SBP": "inputs/diseases/systolic blood pressure-EFO_0006335.tsv",
}

# one colour per disease (validated colour-blind-safe set)
DISEASE_COLORS = {"OC": "#20b2aa", "HTN": "#b71c1c", "SBP": "#2a78d6"}


## Load the network and the two disease gene sets

In [28]:
interactome_df = pd.read_csv(INTERACTOME_FILE)
G = nx.from_pandas_edgelist(interactome_df, "symbol_1", "symbol_2", edge_attr="data_source")

scores = {}
genes = {}
for name in (DISEASE_A, DISEASE_B):
    df = pd.read_csv(DISEASE_FILES[name], sep="\t", low_memory=False)
    df = df[df["globalScore"] >= CUTOFF]
    scores[name] = df.set_index("symbol")["globalScore"]
    genes[name] = set(df["symbol"]) & set(G.nodes())

A, B = genes[DISEASE_A], genes[DISEASE_B]
print(f"{DISEASE_A}: {len(A)} genes, {DISEASE_B}: {len(B)} genes, shared: {len(A & B)}")

OC: 1428 genes, HTN: 1830 genes, shared: 383


## Build the subnetwork
Keep only the disease genes and the links **between them**. Genes with no link to any other
disease gene are kept too: in the HTML views they are lined up in a grid to the right of the network.

In [29]:
sub = G.subgraph(A | B).copy()
isolated = [n for n in sub if sub.degree(n) == 0]


SHARED = f"{DISEASE_A} + {DISEASE_B}"  # genes in both diseases


def group_of(gene):
    if gene in A and gene in B:
        return SHARED
    return f"{DISEASE_A} only" if gene in A else f"{DISEASE_B} only"


print(f"Subnetwork: {sub.number_of_nodes()} genes, {sub.number_of_edges()} links")
print(f"{len(isolated)} of them have no link to another disease gene (kept as unconnected dots)")
pd.Series({n: group_of(n) for n in sub}).value_counts()

Subnetwork: 2875 genes, 16810 links
360 of them have no link to another disease gene (kept as unconnected dots)


HTN only    1447
OC only     1045
OC + HTN     383
Name: count, dtype: int64

## Count links between the groups
How many links stay inside one disease and how many cross between the two diseases?

In [30]:
link_types = pd.Series(
    [" — ".join(sorted((group_of(u), group_of(v)))) for u, v in sub.edges()]
).value_counts()
link_types

OC only — OC only      4984
HTN only — OC only     3864
OC + HTN — OC only     3287
OC + HTN — OC + HTN    1643
HTN only — OC + HTN    1559
HTN only — HTN only    1473
Name: count, dtype: int64

## Interactive HTML views
Open the `.html` files in a browser (needs internet once, to load the vis-network library). Hover a gene to see its group, scores and number of links.
Bigger circle = more links inside this subnetwork. Only genes in both diseases (half/half pies) get a text label, to keep it readable.

**Module layout:** the network is one big connected group, so it is split into **modules** (Louvain communities:
genes linked mostly to each other). Each module is drawn in its own grey circle, named after its 3 best-linked genes.
Links *between* modules are hidden at first: tick **show all links between modules** in the legend, or tick
one module to show only its links to the other modules (hover a module's name for its hub genes).
Small separate pieces share one circle, and genes with no link sit in a grid.
The module of every gene is saved next to each view as `<view>_modules.csv`.

In [31]:
def pie_icon(diseases):
    """SVG pie (as a data URL) with one slice per disease, split by a thin white gap."""
    colors = [DISEASE_COLORS[d] for d in diseases]
    if len(colors) == 1:
        body = f'<circle cx="50" cy="50" r="46" fill="{colors[0]}"/>'
    else:
        body = ""
        step = 360 / len(colors)
        for i, c in enumerate(colors):
            a0, a1 = math.radians(-90 + i * step), math.radians(-90 + (i + 1) * step)
            x0, y0 = 50 + 46 * math.cos(a0), 50 + 46 * math.sin(a0)
            x1, y1 = 50 + 46 * math.cos(a1), 50 + 46 * math.sin(a1)
            large = 1 if step > 180 else 0
            body += (
                f'<path d="M50,50 L{x0:.2f},{y0:.2f} A46,46 0 {large} 1 {x1:.2f},{y1:.2f} Z" '
                f'fill="{c}" stroke="#fcfcfb" stroke-width="5" stroke-linejoin="round"/>'
            )
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="100" height="100" viewBox="0 0 100 100">'
        f'{body}<circle cx="50" cy="50" r="47" fill="none" stroke="#fcfcfb" stroke-width="3"/></svg>'
    )
    return "data:image/svg+xml;charset=utf-8," + quote(svg)


# one icon per group: a plain circle for one disease, a half/half pie for genes in both
GROUP_ICONS_2 = {
    f"{DISEASE_A} only": pie_icon((DISEASE_A,)),
    f"{DISEASE_B} only": pie_icon((DISEASE_B,)),
    SHARED: pie_icon((DISEASE_A, DISEASE_B)),
}

MIN_MODULE = 20  # Louvain modules smaller than this are pooled with the small separate pieces
SPACING = 120  # typical distance from a gene to its nearest neighbour (raise it to spread genes out more)


def ordered_subgraph(graph, genes):
    """Subgraph with genes and links in sorted order. Python's set order changes between runs,
    and Louvain / ForceAtlas2 depend on the order, so this keeps modules and layout the same every run."""
    h = nx.Graph()
    h.add_nodes_from(sorted(genes))
    h.add_edges_from(sorted(tuple(sorted(e)) for e in graph.subgraph(genes).edges()))
    return h


def scaled_layout(h, step, seed):
    """ForceAtlas2 positions (numpy only) of one group, centred on (0, 0) and scaled so the typical distance
    from a gene to its nearest neighbour is `step` (scaling by the outermost genes instead would squeeze
    the dense centre of a module together)."""
    local = nx.forceatlas2_layout(h, seed=seed, max_iter=200) if len(h) > 1 else {n: (0, 0) for n in h}
    xy = np.array([local[n] for n in h], dtype=float)
    xy -= xy.mean(axis=0)
    if len(xy) > 1:
        dist = np.sqrt(((xy[:, None, :] - xy[None, :, :]) ** 2).sum(axis=-1))
        np.fill_diagonal(dist, np.inf)
        factor = step / max(np.median(dist.min(axis=1)), 1e-9)
        # keep the radius between 0.5 and 1.2 x step x sqrt(genes): sparse modules don't blow up,
        # and very dense modules (complexes) aren't crushed
        reach = max(np.sqrt((xy**2).sum(axis=1)).max(), 1e-9)
        size = step * math.sqrt(len(xy)) / reach
        xy *= min(max(factor, 0.5 * size), 1.2 * size)
    return {n: (float(x), float(y)) for n, (x, y) in zip(h, xy)}


def pack_pieces(pieces, gap):
    """Place separate pieces (positions around their own centre) close together, biggest first: each piece
    goes to the first spot on an outward spiral where it stays `gap` away from the pieces already placed."""
    placed, pos = [], {}  # placed: (x, y, radius) of every piece so far
    for local in pieces:
        r = max(math.hypot(x, y) for x, y in local.values())
        t = 0.0
        while True:
            d = gap / (2 * math.pi) * t  # spiral arms are `gap` apart
            cx, cy = d * math.cos(t), d * math.sin(t)
            if all(math.hypot(cx - px, cy - py) >= r + pr + gap for px, py, pr in placed):
                break
            t += min(0.5, gap / 2 / max(d, gap / 2))  # steps of about half a gap along the spiral
        placed.append((cx, cy, r))
        pos.update({n: (cx + x, cy + y) for n, (x, y) in local.items()})
    # centre the packed pieces on the middle of their bounding box
    xs, ys = [x for x, _ in pos.values()], [y for _, y in pos.values()]
    mx, my = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
    return {n: (x - mx, y - my) for n, (x, y) in pos.items()}


def module_layout(graph, isolated_key, sizes=None, find_modules=True, spacing=SPACING, seed=1):
    """Fixed position for every gene, with one area per module.

    The largest connected group is split into modules (Louvain communities: genes linked mostly to each
    other). Each module is laid out on its own inside a circle; small separate pieces (and modules under
    MIN_MODULE genes) share one more circle; genes with no link sit in a grid. The areas are packed in
    rows, biggest first. `sizes` (icon size of each gene) widens the spacing of modules with big icons.
    Returns (positions, module name of each gene, areas to draw in the HTML)."""
    linked = sorted((c for c in nx.connected_components(graph) if len(c) > 1), key=len, reverse=True)
    parts = []
    if linked:
        giant = ordered_subgraph(graph, linked[0])
        parts = nx.community.louvain_communities(giant, seed=seed) if find_modules else [linked[0]]
    modules = sorted((m for m in parts if len(m) >= MIN_MODULE), key=len, reverse=True)
    rest = set().union(*(m for m in parts if len(m) < MIN_MODULE), *linked[1:])
    isolated = sorted((n for n in graph if graph.degree(n) == 0), key=isolated_key)

    # every area: (width, height, name, title, subtitle, local positions around its centre, circle radius)
    shapes = []
    groups = [(m, f"Module {i}") for i, m in enumerate(modules, 1)]
    if rest:
        groups.append((rest, "Small groups"))
    for genes, name in groups:
        h = ordered_subgraph(graph, genes)
        # dense modules (complexes) have high-degree genes with big icons, so space them by icon size too:
        # at least the median icon width plus a 25 px gap, so the icons don't overlap
        step = spacing if sizes is None else max(spacing, 2 * np.median([sizes[n] for n in genes]) + 25)
        if name == "Small groups":
            # the separate pieces are laid out one by one and packed together: one ForceAtlas2 run on all of
            # them pushes the pieces far apart, and the size limit would then squeeze every piece together
            pieces = sorted((ordered_subgraph(h, c) for c in nx.connected_components(h)), key=len, reverse=True)
            local = pack_pieces([scaled_layout(p, step, seed) for p in pieces], step)
        else:
            local = scaled_layout(h, step, seed)
        r = max(math.hypot(x, y) for x, y in local.values()) + 60
        if name == "Small groups":
            subtitle = f"not linked to the main network, or modules under {MIN_MODULE} genes"
        else:
            subtitle = " · ".join(sorted(genes, key=lambda n: (-h.degree(n), n))[:3])  # best-linked genes
        shapes.append((2 * r, 2 * r, name, f"{name} · {len(genes)} genes", subtitle, local, r))
    if isolated:
        dx, dy = 100, 75
        cols = math.ceil(math.sqrt(len(isolated)))
        rows = math.ceil(len(isolated) / cols)
        local = {
            n: ((i % cols - (cols - 1) / 2) * dx, (i // cols - (rows - 1) / 2) * dy)
            for i, n in enumerate(isolated)
        }
        title = f"No link to other genes in this view · {len(isolated)} genes"
        shapes.append(((cols - 1) * dx + 120, (rows - 1) * dy + 120, "No link", title, "", local, None))

    # pack the areas in centred rows; text size grows with the picture so it stays readable when zoomed out
    width = max(max(s[0] for s in shapes), 1.6 * math.sqrt(sum(s[0] * s[1] for s in shapes)))
    font = max(30, width / 70)
    gap, label = 1.5 * font, 2.8 * font
    # make room for the title: about 0.6 font sizes per character (bold), 0.5 for the smaller subtitle
    # (the area's own width is kept at the end, for drawing the grey box)
    shapes = [(max(s[0], 0.6 * font * len(s[3]), 0.5 * font * len(s[4])), *s[1:], s[0]) for s in shapes]
    rows, row = [], []
    for s in shapes:
        if row and sum(t[0] + gap for t in row) + s[0] > width:
            rows.append(row)
            row = []
        row.append(s)
    rows.append(row)

    pos, module_of, areas = {}, {}, []
    y = 0
    for row in rows:
        x = -(sum(s[0] for s in row) + gap * (len(row) - 1)) / 2
        for w, h, name, title, subtitle, local, r, box_w in row:
            cx, cy = x + w / 2, y + label + h / 2
            for n, (lx, ly) in local.items():
                pos[n] = (float(cx + lx), float(cy + ly))
                module_of[n] = name
            areas.append({"name": name, "title": title, "subtitle": subtitle, "x": cx, "y": cy,
                          "r": r, "w": box_w, "h": h, "top": cy - h / 2, "font": font})
            x += w + gap
        y += label + max(s[1] for s in row) + gap
    return pos, module_of, areas


def links_toggle(edges, areas):
    """Legend checkboxes that show / hide the links between modules: one for all of them, and one per module
    for its links to the other modules (nothing if there are no such links)."""
    between = [e for e in edges if e.get("between")]
    if not between:
        return ""
    boxes = "".join(
        f'<label class="links" title="{a["subtitle"]}"><input type="checkbox" data-module="{a["name"]}"> {a["name"]} ({n})</label>'
        for a in areas
        if (n := sum(a["name"] in (e["m1"], e["m2"]) for e in between))
    )
    return (
        f'<label class="links"><input type="checkbox" id="between"> show all links between modules ({len(between)})</label>'
        f'<small>or only the links of a module to the others:</small><div class="mods">{boxes}</div>'
    )


def save_modules(path, module_of, areas, group):
    """Save the module of every gene next to the HTML file (<view>_modules.csv)."""
    hubs = {a["name"]: a["subtitle"] for a in areas if a["name"].startswith("Module")}
    pd.DataFrame(
        {
            "gene": list(module_of),
            "group": [group(n) for n in module_of],
            "module": list(module_of.values()),
            "module_hub_genes": [hubs.get(m, "") for m in module_of.values()],
        }
    ).to_csv(path.with_name(f"{path.stem}_modules.csv"), index=False)


# shared by every HTML view: the network itself, the grey module areas and the "links between modules" switches
NETWORK_JS = """
  const nodes = new vis.DataSet(__NODES__);
  const edges = new vis.DataSet(__EDGES__);
  // module layout: every gene has a fixed position computed in Python, so no physics is needed.
  // Small views without fixed positions (RAS) use physics instead, frozen once the layout has settled.
  const PHYSICS = __PHYSICS__;
  const network = new vis.Network(document.getElementById("net"), {nodes, edges}, {
    nodes: {shape: "image", shapeProperties: {useBorderWithImage: false, interpolation: true},
            font: {size: 22, color: "#0b0b0b", strokeWidth: 3, strokeColor: "#fcfcfb"}},
    edges: {color: {color: "#c9c8c2", highlight: "#52514e"}, width: 1, smooth: false},
    physics: PHYSICS,
    interaction: {hover: true, tooltipDelay: 100}
  });
  // a light background behind each module (circle) and the unlinked genes (box), with its name above
  const AREAS = __AREAS__;
  network.on("beforeDrawing", ctx => {
    ctx.textAlign = "center";
    for (const a of AREAS) {
      ctx.beginPath();
      if (a.r) ctx.arc(a.x, a.y, a.r, 0, 2 * Math.PI);
      else ctx.rect(a.x - a.w / 2, a.y - a.h / 2, a.w, a.h);
      ctx.fillStyle = "#f1f0ec";
      ctx.fill();
      ctx.fillStyle = "#52514e";
      ctx.font = `bold ${a.font}px sans-serif`;
      ctx.fillText(a.title, a.x, a.top - a.font * 1.3);
      ctx.font = `${a.font * 0.8}px sans-serif`;
      ctx.fillText(a.subtitle, a.x, a.top - a.font * 0.3);
    }
  });
  // zoom to everything, with a little room for the area names
  function fitAll() {
    network.fit();
    network.moveTo({scale: network.getScale() * 0.92});
  }
  if (PHYSICS) {
    network.once("stabilizationIterationsDone", () => {
      network.setOptions({physics: false});
      fitAll();
    });
  } else {
    network.once("afterDrawing", fitAll);
  }
  // links between modules are hidden at first (they would cover the picture); the legend switches show
  // all of them, or only the ones that touch a ticked module
  const between = document.getElementById("between");
  const moduleBoxes = [...document.querySelectorAll("#legend input[data-module]")];
  function showBetween() {
    const ticked = new Set(moduleBoxes.filter(b => b.checked).map(b => b.dataset.module));
    edges.update(edges.get({filter: e => e.between}).map(e => ({
      id: e.id, hidden: !(between.checked || ticked.has(e.m1) || ticked.has(e.m2))})));
  }
  if (between) [between, ...moduleBoxes].forEach(b => b.addEventListener("change", showBetween));
"""

HTML_TEMPLATE = """<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>__TITLE__</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/vis-network/9.1.9/standalone/umd/vis-network.min.js"></script>
<style>
  body { margin: 0; font-family: sans-serif; background: #fcfcfb; color: #0b0b0b; }
  #legend { position: absolute; top: 12px; left: 12px; background: #ffffffdd; padding: 8px 12px;
            border: 1px solid #e0dfd9; border-radius: 6px; font-size: 14px; z-index: 1; }
  #legend img { width: 16px; height: 16px; margin-right: 6px; vertical-align: -3px; }
  #legend label.links { display: block; margin: 4px 0; cursor: pointer; user-select: none; }
  #legend .mods { max-height: 35vh; overflow-y: auto; margin-bottom: 4px; }
  #net { width: 100vw; height: 100vh; }
  div.vis-tooltip { white-space: pre-line !important; font-family: sans-serif; }
</style>
</head>
<body>
<div id="legend"><b>__TITLE__</b><br>__LEGEND__<br>__TOGGLE__<small>__COUNTS__ · hover a gene for details</small></div>
<div id="net"></div>
<script>__NETWORK_JS__</script>
</body>
</html>""".replace("__NETWORK_JS__", NETWORK_JS)


def draw(graph, path, title, label_groups=(SHARED,)):
    sizes = {n: 8 + 4 * graph.degree(n) ** 0.5 for n in graph}
    pos, module_of, areas = module_layout(
        graph, isolated_key=lambda n: (list(GROUP_ICONS_2).index(group_of(n)), n), sizes=sizes
    )
    nodes = []
    for n in graph.nodes():
        g = group_of(n)
        lines = [n, f"group: {g}"]
        for d in (DISEASE_A, DISEASE_B):
            s = scores[d].get(n)
            if s is not None:
                lines.append(f"{d} score: {s:.2f}")
        lines.append(f"links here: {graph.degree(n)} | links in full network: {G.degree(n)}")
        lines.append(f"module: {module_of[n]}")
        nodes.append(
            {
                "id": n,
                "label": n if g in label_groups else "",
                "title": "\n".join(lines),
                "image": GROUP_ICONS_2[g],
                # label in the node's colour when the gene belongs to one disease only
                **({"font": {"color": DISEASE_COLORS[g.removesuffix(" only")]}} if g != SHARED else {}),
                "size": sizes[n],
                "x": pos[n][0],
                "y": pos[n][1],
            }
        )
    edges = [
        {
            "from": u,
            "to": v,
            "title": f"{u} – {v}\ndata source: {graph[u][v]['data_source']}",
            **({"between": True, "hidden": True, "m1": module_of[u], "m2": module_of[v]}
               if module_of[u] != module_of[v] else {}),
        }
        for u, v in graph.edges()
    ]

    counts = pd.Series([group_of(n) for n in graph]).value_counts()
    legend = "<br>".join(
        f'<img src="{icon}">{g} ({counts.get(g, 0)})'
        for g, icon in GROUP_ICONS_2.items()
    )
    html = (
        HTML_TEMPLATE.replace("__TITLE__", title)
        .replace("__LEGEND__", legend)
        .replace("__TOGGLE__", links_toggle(edges, areas))
        .replace("__COUNTS__", f"{graph.number_of_nodes()} genes, {graph.number_of_edges()} links")
        .replace("__NODES__", json.dumps(nodes))
        .replace("__EDGES__", json.dumps(edges))
        .replace("__AREAS__", json.dumps(areas))
        .replace("__PHYSICS__", "false")
    )
    with open(path, "w") as f:
        f.write(html)
    save_modules(path, module_of, areas, group_of)
    print(f"Saved {path}: {graph.number_of_nodes()} genes, {graph.number_of_edges()} links, "
          f"{sum(a['name'].startswith('Module') for a in areas)} modules")


# View 1: the whole OC + HTN subnetwork
draw(sub, OUT_DIR / "all_genes.html", f"{DISEASE_A} + {DISEASE_B} disease genes")

Saved outputs/cutoff_0.1/network_OC_HTN/all_genes.html: 2875 genes, 16810 links, 12 modules


### View 2: shared genes and their direct disease-gene neighbours
A smaller, clearer picture: every shared gene plus the OC / HTN genes linked directly to it.
This shows how the shared genes act as **bridges** between the two diseases.

In [32]:
shared = {n for n in sub if group_of(n) == SHARED}
neighbours = {m for n in shared for m in sub.neighbors(n)}
bridge = sub.subgraph(shared | neighbours).copy()

draw(
    bridge,
    OUT_DIR / "shared_bridges.html",
    f"Genes in both {DISEASE_A} and {DISEASE_B} and their neighbours",
    label_groups=(SHARED, f"{DISEASE_B} only"),
)

Saved outputs/cutoff_0.1/network_OC_HTN/shared_bridges.html: 1735 genes, 14175 links, 9 modules


### View 3: only the links that cross between the two diseases
Genes of one disease that directly touch genes of the other disease.

In [33]:
cross_edges = [
    (u, v) for u, v in sub.edges() if group_of(u) != group_of(v) or group_of(u) == SHARED
]
cross = sub.edge_subgraph(cross_edges).copy()

draw(
    cross,
    OUT_DIR / "cross_links.html",
    f"Links crossing {DISEASE_A} and {DISEASE_B}",
    label_groups=(SHARED, f"{DISEASE_A} only", f"{DISEASE_B} only"),
)

Saved outputs/cutoff_0.1/network_OC_HTN/cross_links.html: 2260 genes, 10353 links, 11 modules


---
# Three diseases together: OC, HTN and SBP

Every gene falls into one of 7 groups (OC only, HTN only, SBP only, OC + HTN, OC + SBP, HTN + SBP, all three).
Seven colours can't be told apart reliably, so we use only the **three disease colours** and draw each gene as a
small **pie**: a gene in one disease is a plain circle, a gene in two diseases is half/half, a gene in all three has three slices.

In the HTML view, **click a group in the legend to hide/show it**. Genes in 2+ diseases always have a text label.

In [34]:
DISEASES = ["OC", "HTN", "SBP"]
OUT_DIR_3 = Path("outputs") / f"cutoff_{CUTOFF}" / "network_OC_HTN_SBP"
OUT_DIR_3.mkdir(parents=True, exist_ok=True)

scores3 = {}
genes3 = {}
for name in DISEASES:
    df = pd.read_csv(DISEASE_FILES[name], sep="\t", low_memory=False)
    df = df[df["globalScore"] >= CUTOFF]
    scores3[name] = df.set_index("symbol")["globalScore"]
    genes3[name] = set(df["symbol"]) & set(G.nodes())


def diseases_of(gene):
    """Which of the three diseases this gene belongs to, e.g. ('OC', 'SBP')."""
    return tuple(d for d in DISEASES if gene in genes3[d])


def group_name(gene):
    ds = diseases_of(gene)
    return f"{ds[0]} only" if len(ds) == 1 else " + ".join(ds)


# fixed order of the 7 groups (used for the legend)
GROUPS_3 = [
    "OC only",
    "HTN only",
    "SBP only",
    "OC + HTN",
    "OC + SBP",
    "HTN + SBP",
    "OC + HTN + SBP",
]

sub3 = G.subgraph(set().union(*genes3.values())).copy()
isolated3 = [n for n in sub3 if sub3.degree(n) == 0]

print(
    f"Subnetwork: {sub3.number_of_nodes()} genes, {sub3.number_of_edges()} links "
    f"({len(isolated3)} genes with no link to another disease gene, kept as unconnected dots)"
)
pd.Series([group_name(n) for n in sub3]).value_counts().reindex(GROUPS_3, fill_value=0)

Subnetwork: 3991 genes, 23422 links (485 genes with no link to another disease gene, kept as unconnected dots)


OC only            872
HTN only           748
SBP only          1116
OC + HTN           213
OC + SBP           173
HTN + SBP          699
OC + HTN + SBP     170
Name: count, dtype: int64

## Interactive HTML views
Same module layout as above.

In [35]:
GROUP_ICONS = {g: pie_icon(tuple(g.replace(" only", "").split(" + "))) for g in GROUPS_3}

HTML_TEMPLATE_3 = """<!doctype html>
<html>
<head>
<meta charset="utf-8">
<title>__TITLE__</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/vis-network/9.1.9/standalone/umd/vis-network.min.js"></script>
<style>
  body { margin: 0; font-family: sans-serif; background: #fcfcfb; color: #0b0b0b; }
  #legend { position: absolute; top: 12px; left: 12px; background: #ffffffee; padding: 10px 12px;
            border: 1px solid #e0dfd9; border-radius: 6px; font-size: 14px; z-index: 1; }
  #legend label { display: flex; align-items: center; gap: 6px; margin: 3px 0; cursor: pointer; user-select: none; }
  #legend label.off { opacity: 0.35; }
  #legend input { display: none; }
  #legend label.links input { display: inline; margin: 0; }
  #legend .mods { max-height: 35vh; overflow-y: auto; margin-bottom: 4px; }
  #legend img { width: 16px; height: 16px; }
  #legend small { color: #52514e; }
  #net { width: 100vw; height: 100vh; }
  div.vis-tooltip { white-space: pre-line !important; font-family: sans-serif; }
</style>
</head>
<body>
<div id="legend"><b>__TITLE__</b><br><small>click a group to hide / show it</small>__LEGEND____TOGGLE__<small>__COUNTS__ · hover a gene for details</small></div>
<div id="net"></div>
<script>__NETWORK_JS__
  // legend: click a group to hide / show its genes
  document.querySelectorAll("#legend input[value]").forEach(box => {
    box.addEventListener("change", () => {
      box.parentElement.classList.toggle("off", !box.checked);
      nodes.update(nodes.get({filter: n => n.group_name === box.value})
                        .map(n => ({id: n.id, hidden: !box.checked})));
    });
  });
</script>
</body>
</html>""".replace("__NETWORK_JS__", NETWORK_JS)


def draw3(graph, path, title):
    sizes = {n: 8 + 4 * graph.degree(n) ** 0.5 + (4 if len(diseases_of(n)) > 1 else 0) for n in graph}
    pos, module_of, areas = module_layout(
        graph, isolated_key=lambda n: (GROUPS_3.index(group_name(n)), n), sizes=sizes
    )
    nodes = []
    for n in graph.nodes():
        ds = diseases_of(n)
        lines = [n, f"group: {group_name(n)}"]
        lines += [f"{d} score: {scores3[d].get(n):.2f}" for d in ds]
        lines.append(f"links here: {graph.degree(n)} | links in full network: {G.degree(n)}")
        lines.append(f"module: {module_of[n]}")
        nodes.append(
            {
                "id": n,
                "group_name": group_name(n),
                "label": n if len(ds) > 1 else "",
                "title": "\n".join(lines),
                "image": GROUP_ICONS[group_name(n)],
                # label in the node's colour when the gene belongs to one disease only
                **({"font": {"color": DISEASE_COLORS[ds[0]]}} if len(ds) == 1 else {}),
                "size": sizes[n],
                "x": pos[n][0],
                "y": pos[n][1],
            }
        )
    edges = [
        {
            "from": u,
            "to": v,
            "title": f"{u} – {v}\ndata source: {graph[u][v]['data_source']}",
            **({"between": True, "hidden": True, "m1": module_of[u], "m2": module_of[v]}
               if module_of[u] != module_of[v] else {}),
        }
        for u, v in graph.edges()
    ]

    counts = pd.Series([group_name(n) for n in graph]).value_counts()
    legend = "".join(
        f'<label><input type="checkbox" value="{g}" checked><img src="{GROUP_ICONS[g]}">{g} ({counts.get(g, 0)})</label>'
        for g in GROUPS_3
    )
    html = (
        HTML_TEMPLATE_3.replace("__TITLE__", title)
        .replace("__LEGEND__", legend)
        .replace("__TOGGLE__", links_toggle(edges, areas))
        .replace("__COUNTS__", f"{graph.number_of_nodes()} genes, {graph.number_of_edges()} links")
        .replace("__NODES__", json.dumps(nodes))
        .replace("__EDGES__", json.dumps(edges))
        .replace("__AREAS__", json.dumps(areas))
        .replace("__PHYSICS__", "false")
    )
    with open(path, "w") as f:
        f.write(html)
    save_modules(path, module_of, areas, group_name)
    print(f"Saved {path}: {graph.number_of_nodes()} genes, {graph.number_of_edges()} links, "
          f"{sum(a['name'].startswith('Module') for a in areas)} modules")


# View 1: all genes of the three diseases
draw3(sub3, OUT_DIR_3 / "all_genes.html", "OC + HTN + SBP disease genes")

Saved outputs/cutoff_0.1/network_OC_HTN_SBP/all_genes.html: 3991 genes, 23422 links, 13 modules


### View 2: genes in 2+ diseases and their direct neighbours

In [36]:
multi = {n for n in sub3 if len(diseases_of(n)) > 1}
neighbours3 = {m for n in multi for m in sub3.neighbors(n)}
bridge3 = sub3.subgraph(multi | neighbours3).copy()

draw3(bridge3, OUT_DIR_3 / "shared_bridges.html", "Genes in 2+ diseases and their neighbours")

Saved outputs/cutoff_0.1/network_OC_HTN_SBP/shared_bridges.html: 3109 genes, 22459 links, 16 modules


---
# Renin–angiotensin system (RAS) genes in the network

Where do the 23 RAS genes sit in the interactome, and how close are they to the OC / HTN / SBP disease genes?

The picture shows **every RAS gene** (pie inside a **thick dark ring**) plus every OC / HTN / SBP gene that is
**directly linked** to a RAS gene (plain pies, same colours as above). A RAS gene that is itself a disease gene
has its disease colours inside the ring; a RAS gene in no disease is grey inside the ring.
Links between two RAS genes are drawn dark, all other links light grey.

In [37]:
RAS_GENES = {  # symbol: Entrez ID
    "ACE": 1636,
    "ACE2": 59272,
    "AGT": 183,
    "AGTR1": 185,
    "AGTR2": 186,
    "REN": 5972,
    "ATP6AP2": 10159,
    "MAS1": 4142,
    "MRGPRD": 116512,
    "LNPEP": 4012,
    "ANPEP": 290,
    "ENPEP": 2028,
    "MME": 4311,
    "CMA1": 1215,
    "CTSG": 1511,
    "CPA3": 1359,
    "CTSA": 5476,
    "KLK1": 3816,
    "KLK2": 3817,
    "PRCP": 5547,
    "PREP": 5550,
    "NLN": 57486,
    "THOP1": 7064,
}
OUT_DIR_RAS = Path("outputs") / f"cutoff_{CUTOFF}" / "network_RAS"
OUT_DIR_RAS.mkdir(parents=True, exist_ok=True)

ras = {g for g in RAS_GENES if g in G}
all_disease = set().union(*genes3.values())
missing = sorted(set(RAS_GENES) - ras)
print(
    f"{len(ras)} of {len(RAS_GENES)} RAS genes are in the interactome; missing: {', '.join(missing)}"
)

ras_table = pd.DataFrame(
    [
        {
            "gene": g,
            "entrez_id": RAS_GENES[g],
            "in_interactome": g in G,
            "diseases": " + ".join(diseases_of(g)) if g in G else "",
            "links_to_RAS_genes": len(set(G[g]) & ras) if g in G else None,
            "links_to_disease_genes": len(set(G[g]) & all_disease) if g in G else None,
            "degree_in_full_network": G.degree(g) if g in G else None,
        }
        for g in RAS_GENES
    ]
)
ras_table.to_csv(OUT_DIR_RAS / "ras_genes_summary.csv", index=False)
ras_table

21 of 23 RAS genes are in the interactome; missing: CPA3, MRGPRD


,gene,entrez_id,in_interactome,diseases,links_to_RAS_genes,links_to_disease_genes,degree_in_full_network
0,ACE,1636,True,HTN + SBP,1.0,2.0,5.0
1,ACE2,59272,True,HTN,1.0,3.0,7.0
2,AGT,183,True,HTN + SBP,9.0,6.0,17.0
3,AGTR1,185,True,OC + HTN,2.0,15.0,25.0
4,AGTR2,186,True,,2.0,6.0,10.0
5,REN,5972,True,HTN + SBP,1.0,3.0,8.0
6,ATP6AP2,10159,True,,1.0,1.0,3.0
7,MAS1,4142,True,HTN + SBP,1.0,1.0,1.0
8,MRGPRD,116512,False,,NaN,NaN,NaN
9,LNPEP,4012,True,HTN + SBP,0.0,5.0,7.0


## Build the RAS subnetwork

In [38]:
ras_neighbours = {m for g in ras for m in G[g] if m in all_disease}
ras_net = G.subgraph(ras | ras_neighbours).copy()
# keep only links that touch a RAS gene (links among the neighbours themselves would hide the RAS structure)
ras_net.remove_edges_from(
    [(u, v) for u, v in list(ras_net.edges()) if u not in ras and v not in ras]
)

print(
    f"RAS view: {len(ras)} RAS genes + {len(ras_neighbours - ras)} linked disease genes, "
    f"{ras_net.number_of_edges()} links ({ras_net.subgraph(ras).number_of_edges()} between two RAS genes)"
)

RAS view: 21 RAS genes + 105 linked disease genes, 123 links (12 between two RAS genes)


## Interactive HTML view
Same controls as above: hover a gene for details, click a legend entry to hide / show that group.

In [39]:
from urllib.parse import unquote

RAS_RING = "#0b0b0b"
NO_DISEASE = "#c9c8c2"


def ras_icon(diseases):
    """Disease pie (grey if no disease) inside a thick dark ring that marks a RAS gene."""
    if diseases:  # reuse the pie's SVG, shrunk to fit inside the ring
        pie_svg = unquote(pie_icon(diseases).split(",", 1)[1])
        body = pie_svg.replace(
            '<svg xmlns="http://www.w3.org/2000/svg" width="100" height="100"',
            '<svg x="18" y="18" width="64" height="64"',
            1,
        )
    else:
        body = f'<circle cx="50" cy="50" r="31" fill="{NO_DISEASE}"/>'
    svg = (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="100" height="100" viewBox="0 0 100 100">'
        f'<circle cx="50" cy="50" r="42" fill="#fcfcfb" stroke="{RAS_RING}" stroke-width="10"/>{body}</svg>'
    )
    return "data:image/svg+xml;charset=utf-8," + quote(svg)


# small graph, so a force layout (with a strong pull to the centre, so unlinked RAS genes stay close)
RAS_PHYSICS = """{solver: "barnesHut", barnesHut: {gravitationalConstant: -40000, centralGravity: 2.5,
  springLength: 110, springConstant: 0.02, avoidOverlap: 1}, stabilization: {iterations: 1000}}"""


def draw_ras(graph, path, title):
    nodes = []
    for n in graph.nodes():
        ds = diseases_of(n)
        is_ras = n in ras
        grp = "RAS gene" if is_ras else group_name(n)
        lines = [n, f"RAS gene (Entrez {RAS_GENES[n]})" if is_ras else f"group: {grp}"]
        if is_ras:
            lines.append(f"diseases: {' + '.join(ds) if ds else 'none'}")
        lines += [f"{d} score: {scores3[d].get(n):.2f}" for d in ds]
        lines.append(f"links here: {graph.degree(n)} | links in full network: {G.degree(n)}")
        nodes.append(
            {
                "id": n,
                "group_name": grp,
                "label": n,  # every gene is labelled, shown under its icon
                "title": "\n".join(lines),
                "image": ras_icon(ds) if is_ras else GROUP_ICONS[grp],
                **({"font": {"color": DISEASE_COLORS[ds[0]]}} if len(ds) == 1 else {}),
                "size": (24 if is_ras else 12) + 3 * graph.degree(n) ** 0.5,
            }
        )
    edges = [
        {
            "from": u,
            "to": v,
            "title": f"{u} – {v}\ndata source: {graph[u][v]['data_source']}",
            **({"color": {"color": RAS_RING}, "width": 3} if u in ras and v in ras else {}),
        }
        for u, v in graph.edges()
    ]

    counts = pd.Series([nd["group_name"] for nd in nodes]).value_counts()
    legend_groups = [("RAS gene", ras_icon(()))] + [
        (g, GROUP_ICONS[g]) for g in GROUPS_3 if counts.get(g, 0)
    ]
    legend = "".join(
        f'<label><input type="checkbox" value="{g}" checked><img src="{icon}">{g} ({counts.get(g, 0)})</label>'
        for g, icon in legend_groups
    )
    legend += (
        "<small>inside the ring: the RAS gene's own diseases (grey = none)<br>"
        f"thick dark line = link between two RAS genes<br>not in interactome: {', '.join(missing)}</small><br>"
    )
    html = (
        HTML_TEMPLATE_3.replace("__TITLE__", title)
        .replace("__LEGEND__", legend)
        .replace("__TOGGLE__", "")
        .replace("__COUNTS__", f"{graph.number_of_nodes()} genes, {graph.number_of_edges()} links")
        .replace("__NODES__", json.dumps(nodes))
        .replace("__EDGES__", json.dumps(edges))
        .replace("__AREAS__", "[]")
        .replace("__PHYSICS__", RAS_PHYSICS)
    )
    with open(path, "w") as f:
        f.write(html)
    print(f"Saved {path}: {graph.number_of_nodes()} genes, {graph.number_of_edges()} links")


draw_ras(
    ras_net, OUT_DIR_RAS / "ras_network.html", "RAS genes and their linked OC / HTN / SBP genes"
)


Saved outputs/cutoff_0.1/network_RAS/ras_network.html: 126 genes, 123 links
